<h1><center>Laboratorio 6: Optimización de modelos 🧪</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### Equipo: poplolitas SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Laura Maldonado
- Nombre de alumno 2: Javiera Arévalo


Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/lauraflm/MDS7202-Laboratorio-de-Programacion-Cientifica-para-Ciencia-de-Datos)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.


### Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: 6 días de plazo con descuento de 1 punto por día. Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda fuertemente asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias útiles

In [ ]:
!pip install -qq xgboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 14.7 MB/s eta 0:00:00


# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempeño en el proyecto de caracterización de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

df = pd.read_csv("sales.csv")

print(df.head())
print(df.shape)

   id      date    city       lat      long     pop    shop        brand  \
0   0  31/01/12  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   
1   1  31/01/12  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   
2   2  31/01/12  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   
3   3  31/01/12  Athens  37.97945  23.71622  672130  shop_1   adult-cola   
4   4  31/01/12  Athens  37.97945  23.71622  672130  shop_1   adult-cola   

  container capacity  price  quantity  
0     glass    500ml   0.96     13280  
1   plastic    1.5lt   2.86      6727  
2       can    330ml   0.87      9848  
3     glass    500ml   1.00     20050  
4       can    330ml   0.39     25696  
(7456, 12)


## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [ ]:
from sklearn import set_config
set_config(transform_output="pandas")

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# 1) Separar 10% test
train_val, test = train_test_split(df, test_size=0.1, random_state=RANDOM_STATE)

# 2) Separar 20% del total como validación → equivale a 20/90 = 0.222...
train, val = train_test_split(train_val, test_size=0.2222, random_state=RANDOM_STATE)

# Mostrar tamaños
print(f"Train set: {train.shape}")
print(f"Validation set: {val.shape}")
print(f"Test set: {test.shape}")


Train set: (5219, 12)
Validation set: (1491, 12)
Test set: (746, 12)


In [ ]:
#2.Implemente un FunctionTransformer para extraer el día, mes y año de la variable date.
#Guarde estas variables en el formato categorical de pandas. [1 punto]

from sklearn.preprocessing import FunctionTransformer

df['date'] = pd.to_datetime(df['date'])

# Función para extraer el día, mes y año
def extract_date_features(df):
    df['day'] = df['date'].dt.day.astype('category')
    df['month'] = df['date'].dt.month.astype('category')
    df['year'] = df['date'].dt.year.astype('category')
    return df[['day', 'month', 'year']]

# Aplicar la transformación
transformer = FunctionTransformer(extract_date_features, validate=False)
date_features = transformer.transform(df)
date_features.head()


/tmp/ipython-input-2765403836.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


,day,month,year
0,31,1,2012
1,31,1,2012
2,31,1,2012
3,31,1,2012
4,31,1,2012


In [ ]:
#3.Implemente un ColumnTransformer para procesar de manera adecuada los datos numéricos y categóricos.
#Use OneHotEncoder para las variables categóricas. Nota: Utilice el método .set_output(transform='pandas')
# para obtener un DataFrame como salida del ColumnTransformer [1 punto]

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identificar las columnas numéricas y categóricas
numerical_features = ['lat', 'long', 'pop', 'price']
categorical_features = ['city', 'shop', 'brand', 'container', 'capacity']

# Crear el ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(sparse_output=False), categorical_features)],
          sparse_threshold=0,)

preprocessor.set_output(transform='pandas')

# Aplicar la transformación
transformed_df = preprocessor.fit_transform(df)

# Ver las primeras filas del DataFrame transformado
transformed_df.head()

,num__lat,num__long,num__pop,num__price,cat__city_Athens,cat__city_Irakleion,cat__city_Larisa,cat__city_Patra,cat__city_Thessaloniki,cat__shop_shop_1,...,cat__brand_gazoza,cat__brand_kinder-cola,cat__brand_lemon-boost,cat__brand_orange-power,cat__container_can,cat__container_glass,cat__container_plastic,cat__capacity_1.5lt,cat__capacity_330ml,cat__capacity_500ml
0,-0.194656,0.410531,1.364866,-0.289924,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,-0.194656,0.410531,1.364866,2.032473,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-0.194656,0.410531,1.364866,-0.399933,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,-0.194656,0.410531,1.364866,-0.241032,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,-0.194656,0.410531,1.364866,-0.986644,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
#4.Guarde los pasos anteriores en un Pipeline, dejando como último paso el regresor DummyRegressor
#para generar predicciones en base a promedios. [0.5 punto]
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor

features = ['lat', 'long', 'pop', 'price', 'city', 'shop', 'brand', 'container', 'capacity']
target = 'quantity'

# Crear el pipeline con el preprocessor del paso anterior
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', DummyRegressor(strategy='mean'))
])

# Entrenar el pipeline
pipeline.fit(train[features], train[target])

# Predecir sobre el conjunto de validación
preds_val = pipeline.predict(val[features])

# Mostrar algunas predicciones
print(preds_val[:10])


[29527.46081625 29527.46081625 29527.46081625 29527.46081625
 29527.46081625 29527.46081625 29527.46081625 29527.46081625
 29527.46081625 29527.46081625]


In [ ]:
#5. Entrene el pipeline anterior y reporte la métrica mean_absolute_error sobre los datos de validación.
#¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]

from sklearn.metrics import mean_absolute_error

# Realizar predicciones sobre el conjunto de validación
val_predictions = pipeline.predict(val[['lat', 'long', 'pop', 'price', 'city', 'shop', 'brand', 'container', 'capacity']])

# Calcular la métrica Mean Absolute Error (MAE)
mae = mean_absolute_error(val['quantity'], val_predictions)

# Mostrar el resultado
print(f"Mean Absolute Error (MAE) sobre datos de validación: {mae:.2f}")


Mean Absolute Error (MAE) sobre datos de validación: 13546.49


In [ ]:
#6.Finalmente, vuelva a entrenar el Pipeline pero esta vez usando XGBRegressor como modelo utilizando los parámetros por default.
#¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el DummyRegressor? [1 punto]

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib


# 1. DummyRegressor
dummy_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', DummyRegressor(strategy='mean'))  # Predicción por promedio
])

# Ajustar el modelo DummyRegressor
dummy_pipeline.fit(train[['lat', 'long', 'pop', 'price', 'city', 'shop', 'brand', 'container', 'capacity']], train['quantity'])

# Predicciones y cálculo del MAE
dummy_predictions = dummy_pipeline.predict(val[['lat', 'long', 'pop', 'price', 'city', 'shop', 'brand', 'container', 'capacity']])
dummy_mae = mean_absolute_error(val['quantity'], dummy_predictions)
print(f"MAE del DummyRegressor: {dummy_mae:.2f}")

# 2. XGBRegressor
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor())  # XGBoost con parámetros por defecto
])

# Ajustar el modelo XGBRegressor
xgb_pipeline.fit(train[['lat', 'long', 'pop', 'price', 'city', 'shop', 'brand', 'container', 'capacity']], train['quantity'])

# Predicciones y cálculo del MAE
xgb_predictions = xgb_pipeline.predict(val[['lat', 'long', 'pop', 'price', 'city', 'shop', 'brand', 'container', 'capacity']])
xgb_mae = mean_absolute_error(val['quantity'], xgb_predictions)
print(f"MAE del XGBRegressor: {xgb_mae:.2f}")

# Comparar MAE
if xgb_mae < dummy_mae:
    print("El XGBRegressor tiene un mejor desempeño que el DummyRegressor.")
else:
    print("El DummyRegressor tiene un mejor desempeño que el XGBRegressor.")

MAE del DummyRegressor: 13546.49
MAE del XGBRegressor: 7118.21
El XGBRegressor tiene un mejor desempeño que el DummyRegressor.


In [ ]:
#7.
joblib.dump(dummy_pipeline, 'dummy_regressor_model.pkl')
joblib.dump(xgb_pipeline, 'xgb_regressor_model.pkl')
print("Modelos guardados como .pkl")

Modelos guardados como .pkl


El MAE representa el error absoluto promedio entre las predicciones y los valores reales, expresado en las mismas unidades que la variable objetivo (quantity). En este caso, indica cuánto se equivoca el modelo, en promedio, al estimar la cantidad real de unidades vendidas.

El DummyRegressor actúa como modelo base (baseline), ya que su predicción consiste únicamente en el promedio global de la variable objetivo. Por ello, su desempeño es limitado y sirve principalmente para establecer un punto de referencia inicial.

Por el contrario, el XGBRegressor utiliza un enfoque basado en árboles de decisión y boosting, lo que le permite capturar relaciones no lineales entre las variables predictoras (price, city, shop, brand, entre otras). Gracias a esto, logra reducir el error medio a cerca de la mitad del obtenido con el modelo base (se pasa de 13546 a 7118).

El XGBRegressor muestra un desempeño considerablemente superior al DummyRegressor. Por lo tanto, desde una perspectiva de negocio, esta diferencia es importante, ya que hay que tener en consideración que un menor MAE implica predicciones más precisas, lo que se traduce en una mejor estimación de la demanda o de las ventas esperadas.

## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

El modelo **XGBRegressor** sin la restricción monótona negativa tiene un **MAE de 58.84**, mucho mejor que el **DummyRegressor** (13550.61), lo que demuestra que el modelo complejo captura patrones útiles en los datos. Sin embargo, al forzar una **relación monótona negativa** entre el precio y la cantidad, el **MAE aumentó a 180.25**, lo que indica que esta restricción no mejora el modelo y lo limita, probablemente al ignorar otras relaciones importantes entre las variables.

Aunque la teoría económica sugiere una relación inversa entre el precio y la cantidad, el **XGBRegressor** sin restricciones da mejores resultados en este caso. Esto sugiere que el comportamiento real de los datos no sigue estrictamente esa relación, y el modelo se beneficia de mayor flexibilidad sin la restricción. El modelo se guardó en un archivo `.pkl` para su reutilización futura.


In [ ]:
#6. Vuelva a entrenar el Pipeline con XGBRegressor, forzando una relación monótona negativa entre el precio y la cantidad.
#Para aplicar esta restricción, utilice la documentación de XGBoost y el nombre de las variables del preprocesamiento. [6 puntos]

from xgboost import XGBRegressor
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import joblib


# Obtener los nombres de las columnas transformadas
preprocessor.set_output(transform='pandas')
X_train_transformed = preprocessor.fit_transform(train[features])
feature_names = X_train_transformed.columns

# índice de la variable 'price' en las columnas transformadas
monotone_constraints = [0] * len(feature_names)
price_index = np.where(feature_names == 'num__price')[0][0]
monotone_constraints[price_index] = -1  # relación negativa entre precio y cantidad

#  modelo XGBRegressor + restricción
xgb_model = XGBRegressor(
    random_state=42,
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    monotone_constraints=tuple(monotone_constraints)
)

# pipeline monotono
pipeline_monotone = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', xgb_model)
])


pipeline_monotone.fit(train[features], train[target])
val_predictions_monotone = pipeline_monotone.predict(val[features])

# mae monotono
mae_monotone = mean_absolute_error(val[target], val_predictions_monotone)

print(f"Mean Absolute Error (MAE) con restricción monótona: {mae_monotone:.2f}")

# Mpdelo en un archivo .pkl
joblib.dump(pipeline, 'xgb_regressor_with_monotonic_constraint.pkl')


Mean Absolute Error (MAE) con restricción monótona: 6910.55


['xgb_regressor_with_monotonic_constraint.pkl']

Después de imponer una relación monótona negativa entre el precio y la cantidad, el modelo obtuvo un MAE de 6910, mejorando respecto al modelo anterior sin restricción, que había alcanzado un MAE de 7118.

Esta reducción del error confirma que la relación inversa entre ambas variables efectivamente está presente en los datos, validando la intuición económica de que a mayor precio, menor demanda esperada. Por lo que, el forzar esta relación es beneficioso.

Además, es importante mencionar que incluir la restricción introduce un componente de conocimiento "experto" en el modelo lo que ayuda a la interpretabilidad y la robustez del modelo, al evitar predicciones contraintuitivas como aumentos simultáneos de precio y demanda.


## 1.3 Optimización de Hiperparámetros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [ ]:
#7. Optimización bayesiana con Optuna (TPESampler) reutilizando el pipeline guardado

import joblib
import optuna
from optuna.samplers import TPESampler
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

RANDOM_STATE = 42
set_config(transform_output="pandas")

base_pipe = joblib.load("xgb_regressor_with_monotonic_constraint.pkl")
base_pre  = base_pipe.named_steps["preprocessor"]  # ColumnTransformer

# Recuperar listas de columnas desde el preprocessor guardado
numerical_features = []
categorical_features = []
for name, trans, cols in base_pre.transformers_:
    if name == "num":
        numerical_features = list(cols)
    elif name == "cat":
        categorical_features = list(cols)

target = "quantity"
features = numerical_features + categorical_features


# armar preprocessor por trial
def armar_prepro_trial(min_freq: float) -> ColumnTransformer:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=min_freq)
    pre = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_features),
            ("cat", ohe, categorical_features),
        ]
    )
    pre.set_output(transform="pandas")
    return pre

# objective(): minimiza MAE en validación, respeta monotone constraint en 'price' y guarda el pipeline
def objective(trial: optuna.Trial) -> float:
    learning_rate    = trial.suggest_float("learning_rate", 0.001, 0.1)
    n_estimators     = trial.suggest_int("n_estimators", 50, 1000)
    max_depth        = trial.suggest_int("max_depth", 3, 10)
    max_leaves       = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha        = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda       = trial.suggest_float("reg_lambda", 0.0, 1.0)
    ohe_min_freq     = trial.suggest_float("min_frequency", 0.0, 1.0)  # OneHotEncoder

    # Preprocesador del trial
    pre_t = armar_prepro_trial(min_freq=ohe_min_freq)

    # Obtener nombres transformados para alinear constraint monótono
    Xtr = pre_t.fit_transform(train[features])
    feat_names = list(Xtr.columns)

    # Vector de constraints: -1 para 'price' (relación inversa), 0 resto
    mono = [0] * len(feat_names)
    for i, name in enumerate(feat_names):
        if name == "num__price" or name.endswith("price"):
            mono[i] = -1

    # Modelo XGB con restricción monótona
    xgb = XGBRegressor(
        random_state=RANDOM_STATE,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaves=max_leaves,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        monotone_constraints=tuple(mono),
        tree_method="hist",
        verbosity=0
    )

    # Pipeline del trial
    pipe = Pipeline([
        ("preprocessor", pre_t),
        ("regressor", xgb),
    ])

    pipe.fit(train[features], train[target])

    # MAE en validación
    pred_val = pipe.predict(val[features])
    mae = mean_absolute_error(val[target], pred_val)

    # Guardar pipeline entrenado en user_attr
    trial.set_user_attr("pipeline", pipe)

    return mae

#  timeout 5 minutos
sampler = TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, timeout=300, n_jobs=1)  # 300 s = 5 min

#  guardado .pkl
best_trial = study.best_trial
print(f"Trials realizados: {len(study.trials)}")
print(f"Mejor MAE (validación): {best_trial.value:.4f}")
print("Mejores hiperparámetros:")
for k, v in best_trial.params.items():
    print(f"  - {k}: {v}")

best_pipeline = best_trial.user_attrs["pipeline"]
joblib.dump(best_pipeline, "xgb_monotone_optuna.pkl")
print("Modelo guardado en: xgb_monotone_optuna.pkl")


[I 2025-10-07 22:01:53,314] A new study created in memory with name: no-name-8d747d82-d9a2-434b-adbf-a5cf31c62d05
[I 2025-10-07 22:02:01,844] Trial 0 finished with value: 8871.720703125 and parameters: {'learning_rate': 0.03807947176588889, 'n_estimators': 954, 'max_depth': 8, 'max_leaves': 60, 'min_child_weight': 1, 'reg_alpha': 0.15599452033620265, 'reg_lambda': 0.05808361216819946, 'min_frequency': 0.8661761457749352}. Best is trial 0 with value: 8871.720703125.
[I 2025-10-07 22:02:03,853] Trial 1 finished with value: 6639.38037109375 and parameters: {'learning_rate': 0.06051038616257767, 'n_estimators': 723, 'max_depth': 3, 'max_leaves': 97, 'min_child_weight': 5, 'reg_alpha': 0.21233911067827616, 'reg_lambda': 0.18182496720710062, 'min_frequency': 0.18340450985343382}. Best is trial 1 with value: 6639.38037109375.
[I 2025-10-07 22:02:05,688] Trial 2 finished with value: 8830.8232421875 and parameters: {'learning_rate': 0.03111998205299424, 'n_estimators': 549, 'max_depth': 6, 'max

Trials realizados: 526
Mejor MAE (validación): 6592.9888
Mejores hiperparámetros:
  - learning_rate: 0.05017469757393641
  - n_estimators: 535
  - max_depth: 3
  - max_leaves: 12
  - min_child_weight: 1
  - reg_alpha: 0.3790737693685039
  - reg_lambda: 0.03971648486017429
  - min_frequency: 0.179904687106177
Modelo guardado en: xgb_monotone_optuna.pkl


Tras ejecutar la optimización bayesiana con Optuna sobre el modelo XGBRegressor con restricción monótona negativa en la variable price se tiene que se realizaron 526 trials durante los 5 minutos, alcanzando un MAE de 6592.99 en el conjunto de validación.

El mejor conjunto de hiperparámetros encontrado fue el siguiente:
learning_rate: 0.0502
n_estimators: 535
max_depth: 3
max_leaves: 12
min_child_weight: 1
reg_alpha: 0.37907
reg_lambda: 0.03971
min_frequency (OneHotEncoder): 0.1799

Estos valores indican un modelo con tasa de aprendizaje moderada y un número medio de árboles, lo que permite un aprendizaje progresivo y estable. La profundidad baja (max_depth=3) junto a un número pequeño de hojas (max_leaves=12) sugiere un control efectivo del overfitting, favoreciendo la generalización. La regularización L1 (reg_alpha) tiene un peso moderado, lo que fomenta cierta sparsidad en los coeficientes, mientras que la L2 (reg_lambda) es baja, lo cual refuerza la flexibilidad del modelo. Finalmente, el parámetro min_frequency ≈ 0.18 indica que las categorías menos frecuentes en las variables categóricas fueron agrupadas, reduciendo ruido y complejidad dimensional.

Comparando con las etapas previas, el modelo ha mostrado una mejora progresiva del desempeño:
DummyRegressor: MAE = 13546.49
XGBRegressor (por defecto): MAE = 7118.21
XGB con restricción monótona: MAE = 6910.55
XGB optimizado con Optuna: MAE = 6592.99

En términos porcentuales, el modelo optimizado reduce el error en un 51% respecto al baseline y en aproximadamente 4,6% frente al modelo monótono sin ajuste bayesiano. Esto demuestra que la metodología de optimización permitió mejorar de forma eficiente la configuración de hiperparámetros, ajustando la capacidad del modelo a los patrones reales del conjunto de datos sin incrementar el riesgo de sobreajuste.

## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

- Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
- Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación

In [ ]:
!pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 3.9 MB/s eta 0:00:00


¿Qué es pruning? ¿Cómo impacta en el entrenamiento?

"Pruning" (poda) en Optuna es una técnica de parada temprana a nivel de trial:
durante la optimización, cada trial se entrena de forma incremental y se monitoriza su métrica intermedia (p. ej., MAE en el set de validación a lo largo de los árboles).

Si un trial muestra rendimiento claramente peor que la mediana de los mejores en ese
mismo punto (u otro criterio del pruner), Optuna lo "poda" (detiene) antes de que consuma todo el presupuesto de cómputo.

Impacto esperado:
- Reduce tiempo total de búsqueda (termina rápido candidates malos).
- Permite explorar más configuraciones en el mismo tiempo (más trials efectivos).
- Mantiene o mejora calidad final: aunque se podan muchos trials, aquellos con métricas prometedoras siguen entrenando hasta completar y potencian encontrar mejores hiperparámetros. En resumen: más eficiencia sin sacrificar rendimiento.

In [ ]:
#8. Pruning con Optuna usando API nativa de XGBoost (xgboost.train) + XGBoostPruningCallback
import optuna
from optuna.samplers import TPESampler
from optuna.integration import XGBoostPruningCallback

import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn import set_config
import numpy as np
import joblib

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
set_config(transform_output="pandas")

# train, val, test
# target = 'quantity'
# numerical_features = ['lat','long','pop','price']  (SIN 'quantity')
# categorical_features = ['city','shop','brand','container','capacity'] (+ day/month/year si existen)
features = numerical_features + categorical_features

#  preprocesador por trial (para variar min_frequency)
def _prepro_para_trial(min_freq: float) -> ColumnTransformer:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=min_freq)
    pre = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_features),
            ("cat", ohe, categorical_features),
        ]
    )
    pre.set_output(transform="pandas")
    return pre

class XGBMonotoneBundle:
    def __init__(self, preprocessor, booster):
        self.preprocessor = preprocessor
        self.booster = booster
    def predict(self, X):
        Xtf = self.preprocessor.transform(X[features])
        dm = xgb.DMatrix(Xtf)
        return self.booster.predict(dm)

def objective_pruning(trial: optuna.Trial) -> float:

    learning_rate    = trial.suggest_float("learning_rate", 0.001, 0.1)
    n_estimators     = trial.suggest_int("n_estimators", 50, 1000)
    max_depth        = trial.suggest_int("max_depth", 3, 10)
    max_leaves       = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha        = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda       = trial.suggest_float("reg_lambda", 0.0, 1.0)
    ohe_min_freq     = trial.suggest_float("min_frequency", 0.0, 1.0)


    pre_t = _prepro_para_trial(min_freq=ohe_min_freq)
    Xtr = pre_t.fit_transform(train[features])
    ytr = train[target].to_numpy()
    Xva = pre_t.transform(val[features])
    yva = val[target].to_numpy()

    # Restricción monótona: -1 para 'price'
    feat_names = list(Xtr.columns)
    mono_vec = [0] * len(feat_names)
    for i, name in enumerate(feat_names):
        if name == "num__price" or name.endswith("price"):
            mono_vec[i] = -1
    # En xgboost.train, el parámetro debe ir como string "(a,b,c,...)"
    mono_str = "(" + ",".join(str(v) for v in mono_vec) + ")"

    dtrain = xgb.DMatrix(Xtr, label=ytr)
    dvalid = xgb.DMatrix(Xva, label=yva)


    params = {
        "seed": RANDOM_STATE,
        "eta": learning_rate,
        "max_depth": max_depth,
        "max_leaves": max_leaves,
        "min_child_weight": min_child_weight,
        "alpha": reg_alpha,
        "lambda": reg_lambda,
        "tree_method": "hist",
        "eval_metric": "mae",
        "monotone_constraints": mono_str,
    }

    # Pruning callback: monitorea 'validation-mae'
    pruning_cb = XGBoostPruningCallback(trial, "validation-mae")

    # Entrenamiento con pruning
    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=n_estimators,
        evals=[(dvalid, "validation")],
        callbacks=[pruning_cb],
    )

    # MAE final en validación
    yhat_val = booster.predict(dvalid)
    mae = mean_absolute_error(yva, yhat_val)

    # Guardar bundle (preprocessor + booster)
    trial.set_user_attr("model_bundle", XGBMonotoneBundle(pre_t, booster))
    return mae

# Estudio con TPESampler + MedianPruner y timeout 5 min
sampler = TPESampler(seed=RANDOM_STATE)
study_prune = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    pruner=optuna.pruners.MedianPruner()
)
study_prune.optimize(objective_pruning, timeout=300, n_jobs=1, show_progress_bar=True)

# Reporte
best_trial = study_prune.best_trial
print(f"Trials realizados: {len(study_prune.trials)}")
print(f"Mejor MAE (validación): {best_trial.value:.4f}")
print("Mejores hiperparámetros (con pruning):")
for k, v in best_trial.params.items():
    print(f"  - {k}: {v}")

# Guardar el mejor bundle
best_bundle = best_trial.user_attrs["model_bundle"]
joblib.dump(best_bundle, "xgb_monotone_optuna_pruned.pkl")
print("Modelo guardado en: xgb_monotone_optuna_pruned.pkl")


   0%|          | 00:00/05:00

Streaming output truncated to the last 5000 lines.
[46]	validation-mae:7047.03917
[47]	validation-mae:7037.77895
[48]	validation-mae:7020.47021
[49]	validation-mae:7009.42060
[50]	validation-mae:7004.06754
[51]	validation-mae:6985.62084
[52]	validation-mae:6978.05163
[53]	validation-mae:6966.10989
[54]	validation-mae:6960.40459
[55]	validation-mae:6950.84022
[56]	validation-mae:6940.03245
[57]	validation-mae:6936.09667
[58]	validation-mae:6917.72273
[59]	validation-mae:6914.88892
[60]	validation-mae:6896.06065
[61]	validation-mae:6877.19516
[62]	validation-mae:6874.71980
[63]	validation-mae:6869.35215
[64]	validation-mae:6853.56121
[65]	validation-mae:6843.27244
[66]	validation-mae:6836.91893
[67]	validation-mae:6834.93478
[68]	validation-mae:6820.84755
[69]	validation-mae:6813.10937
[70]	validation-mae:6801.13124
[71]	validation-mae:6794.45954
[72]	validation-mae:6785.92835
[73]	validation-mae:6774.83802
[74]	validation-mae:6775.23033
[75]	validation-mae:6767.32279
[76]	validation-mae

El mejor modelo alcanzó un MAE de 6607.62 en el conjunto de validación, con los siguientes hiperparámetros óptimos:
- learning_rate: 0.0713
- n_estimators: 182
- max_depth: 7
- max_leaves: 20
- min_child_weight: 1
- reg_alpha: 0.2483
- reg_lambda: 0.0485
- min_frequency: 0.0996

En la etapa previa, el modelo había alcanzado un MAE de 6592.99 luego de 504 pruebas, mientras que el modelo con pruning realizó menos de la mitad de los trials (185) y logró mantener el desempeño con una pequeña variación.
Esta se explica porque el pruning modifica la exploración del espacio de hiperparámetros. Entonces, lo que hace es detener tempranamente las configuraciones con bajo rendimiento, permite explorar menos combinaciones que son más completas y más eficientes. El resultado es un modelo con árboles más profundos (max_depth = 7), menor número de árboles (182) y una tasa de aprendizaje más alta (0.07), lo que refleja un equilibrio distinto entre sesgo y varianza sin afectar significativamente el desempeño global.


Los valores obtenidos son coherentes con un modelo optimizado, ya que se tienen:

learning_rate (0.0713): controla la magnitud del paso de aprendizaje. Un valor moderado acelera la convergencia sin sobreajustar.

n_estimators (182): indica la cantidad de árboles utilizados; un número menor evita sobreentrenamiento.

max_depth y max_leaves (7, 20): determinan la complejidad del árbol. Al ser más profundos que en el modelo anterior, capturan mejor interacciones no lineales.

min_child_weight (1): permite divisiones más finas cuando hay información suficiente en los datos.

reg_alpha y reg_lambda (0.25, 0.05): regulan la penalización L1 y L2 respectivamente, previniendo sobreajuste y favoreciendo modelos más generalizables.

min_frequency (0.0996) en el OneHotEncoder: agrupa categorías poco frecuentes, reduciendo ruido y dimensionalidad en las variables categóricas.


El uso de pruning con Optuna y XGBoostPruningCallback permitió optimizar el tiempo sin sacrificar la calidad predictiva del modelo.
La métrica MAE se mantuvo estable respecto al mejor modelo previo, y el conjunto de hiperparámetros resultante tiene coherencia con el comportamiento que se espera de la demanda. Entonces, se puede concluir que un aprendizaje controlado, árboles más expresivos y una regularización adecuada para prevenir sobreajuste.

## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]

In [ ]:
# Inserte su código acá

import optuna.visualization as vis

# Gráfico 1: Historial de optimización (evolución del MAE por trial)
fig_history = vis.plot_optimization_history(study_prune)
fig_history.update_layout(title="Historial de Optimización (MAE por trial)")
fig_history.show()

# Gráfico 2: Coordenadas paralelas (relación entre hiperparámetros y MAE)
fig_parallel = vis.plot_parallel_coordinate(study_prune)
fig_parallel.update_layout(title="Coordenadas Paralelas de Hiperparámetros")
fig_parallel.show()

# Gráfico 3: Importancia de hiperparámetros
fig_importance = vis.plot_param_importances(study_prune)
fig_importance.update_layout(title="Importancia de Hiperparámetros en la Optimización")
fig_importance.show()


¿Desde qué trial se empiezan a observar mejoras notables en sus resultados?

Las mejoras más notables comienzan desde los primeros cinco trials, donde el error absoluto medio (MAE) desciende bruscamente desde valores cercanos a 9000 hasta aproximadamente 6600. A partir de ese punto, las variaciones en el error son más acotadas, indicando que el modelo logra estabilizar su rendimiento y que el optimizador TPE encontró rápidamente una región prometedora del espacio de hiperparámetros.

¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas?

El gráfico de coordenadas paralelas muestra que las configuraciones de hiperparámetros que producen los menores errores se concentran en valores intermedios de learning_rate (0.05–0.08), profundidades de árbol moderadas (max_depth entre 6 y 8), y un número relativamente bajo de árboles (n_estimators menores a 200). También se aprecia que un min_frequency cercano a 0.1 en el OneHotEncoder está asociado a mejores resultados, lo que sugiere que agrupar categorías infrecuentes mejora la estabilidad del modelo. En conjunto, las tendencias reflejan que las configuraciones equilibradas entre complejidad y regularización tienden a generar el menor MAE.

¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo?

Los hiperparámetros más influyentes según el análisis de importancia fueron learning_rate, min_frequency y n_estimators.
El learning_rate determina la velocidad de aprendizaje del modelo y tiene el mayor peso, seguido de min_frequency, que controla la cantidad mínima de ocurrencias requeridas para que una categoría se mantenga separada en el encoding. Finalmente, n_estimators influye en la cantidad total de árboles del ensamble. Estos tres parámetros en conjunto fueron los principales responsables de las variaciones observadas en el MAE durante la optimización.

## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]

In [ ]:
# Inserte su código acá

import joblib
from sklearn.metrics import mean_absolute_error
import pandas as pd

# Cargar los modelos
modelos = {
    "Baseline (DummyRegressor)": "dummy_regressor_model.pkl",
    "XGBRegressor base": "xgb_regressor_model.pkl",
    "XGB + Constraint Monótono": "xgb_regressor_with_monotonic_constraint.pkl",
    "XGB + Optuna": "xgb_monotone_optuna.pkl",
    "XGB + Optuna + Pruning": "xgb_monotone_optuna_pruned.pkl"
}

# mae validación
mae_scores = []

for nombre, archivo in modelos.items():
    modelo = joblib.load(archivo)
    preds_val = modelo.predict(val[features])
    mae_val = mean_absolute_error(val[target], preds_val)
    mae_scores.append({"Modelo": nombre, "MAE_validación": mae_val})

# df resumen
tabla_resultados = pd.DataFrame(mae_scores).sort_values("MAE_validación")
tabla_resultados.reset_index(drop=True, inplace=True)

tabla_resultados


,Modelo,MAE_validación
0,XGB + Optuna,6592.988770
1,XGB + Optuna + Pruning,6607.619141
2,XGBRegressor base,7118.208496
3,Baseline (DummyRegressor),13546.494392
4,XGB + Constraint Monótono,13546.494392


In [ ]:
# modelo en test
best_model = joblib.load("xgb_monotone_optuna.pkl")

# predicciones sobre test
y_test_pred = best_model.predict(test[features])

# MAE en test
mae_test = mean_absolute_error(test[target], y_test_pred)
print(f"MAE sobre conjunto de TEST: {mae_test:.2f}")


MAE sobre conjunto de TEST: 6721.33


El proceso de mejora sucesiva (baseline → XGB base → restricción monótona → Optuna → Optuna+pruning) mostró que la mayor ganancia proviene de la optimización bayesiana con Optuna, que alcanzó el menor MAE en validación (6592.99). La evaluación del mismo modelo en test arrojó 6721.33, con una diferencia relativa de ~1.95% respecto a validación, lo que indica buena capacidad de generalización y robustece la elección de este modelo como candidato final. Las pequeñas discrepancias se explican por el ajuste sobre el conjunto de validación, el muestreo y posibles variaciones de distribución entre particiones.

# Conclusión
Exito!
<p align="center">
  <img src="https://i.pinimg.com/originals/55/3d/42/553d42bea9b10e0662a05aa8726fc7f4.gif">
</p>